# RAG Pipeline :- Data Ingestion To Vector DB store


In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/95/00z95nwx2vggmnw4dygw43f80000gn/T/ipykernel_35572/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/Users/varun/Downloads/Ragging/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def processing_pdfs(pdf_directory):                # Read all the pdf's inside the directory
    
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))      # Find all PDF files recursively
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:                               # Add source information to metadata
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"✅ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"❌ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = processing_pdfs("../data")                  # Process all PDFs in the data directory

Found 4 PDF files to process

Processing: The Ultimate Python Handbook.pdf
✅ Loaded 61 pages

Processing: Python Programming.pdf
✅ Loaded 143 pages

Processing: 2312.10997v5.pdf
✅ Loaded 21 pages

Processing: NEET 2024 Paper - Chemistry.pdf
✅ Loaded 12 pages

Total documents loaded: 237


In [7]:
all_pdf_documents

[Document(metadata={'producer': '3.0.10 (5.0.18)', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-06-16T02:12:11+05:30', 'author': 'Harry Codes', 'moddate': '2024-06-15T22:44:10+02:00', 'source': '../data/pdf_files/The Ultimate Python Handbook.pdf', 'total_pages': 61, 'page': 0, 'page_label': '1', 'source_file': 'The Ultimate Python Handbook.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': '3.0.10 (5.0.18)', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-06-16T02:12:11+05:30', 'author': 'Harry Codes', 'moddate': '2024-06-15T22:44:10+02:00', 'source': '../data/pdf_files/The Ultimate Python Handbook.pdf', 'total_pages': 61, 'page': 1, 'page_label': '2', 'source_file': 'The Ultimate Python Handbook.pdf', 'file_type': 'pdf'}, page_content='1 \n \nPREFACE  \nWelcome to the “Ultimate Python Programming Handbook," your comprehensive guide to \nmastering Python programming. This handbook is designed for beginners and any







## Splitting Data into Chunks

In [8]:
def splitting_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    
    """Splitting documents into small chunks for improving the overall RAG speed & performance"""
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function  = len,
        separators= ["\n\n", "\n", " ", ""]
    )
    
    split_docs = splitter.split_documents(documents)        ###split_documents is a function just like in loader.loaad() here .load is a function so is split_document here is 
    
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    #Example of a chunk 
    
    if split_docs:
        print(f"Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"MetaData: {split_docs[0].metadata}")
        
    return split_docs

In [9]:
chunks= splitting_documents(all_pdf_documents)
chunks

Split 237 documents into 515 chunks
Example chunk:
Content: 1 
 
PREFACE  
Welcome to the “Ultimate Python Programming Handbook," your comprehensive guide to 
mastering Python programming. This handbook is designed for beginners and anyone looking to 
strength...
MetaData: {'producer': '3.0.10 (5.0.18)', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-06-16T02:12:11+05:30', 'author': 'Harry Codes', 'moddate': '2024-06-15T22:44:10+02:00', 'source': '../data/pdf_files/The Ultimate Python Handbook.pdf', 'total_pages': 61, 'page': 1, 'page_label': '2', 'source_file': 'The Ultimate Python Handbook.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': '3.0.10 (5.0.18)', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-06-16T02:12:11+05:30', 'author': 'Harry Codes', 'moddate': '2024-06-15T22:44:10+02:00', 'source': '../data/pdf_files/The Ultimate Python Handbook.pdf', 'total_pages': 61, 'page': 1, 'page_label': '2', 'source_file': 'The Ultimate Python Handbook.pdf', 'file_type': 'pdf'}, page_content='1 \n \nPREFACE  \nWelcome to the “Ultimate Python Programming Handbook," your comprehensive guide to \nmastering Python programming. This handbook is designed for beginners and anyone looking to \nstrengthen their foundational knowledge of Python, a versatile and user-friendly programming \nlanguage. \nPURPOSE AND AUDIENCE  \nThis handbook aims to make programming accessible and enjoyable for everyone. Whether \nyou\'re a student new to coding, a professional seeking to enhance your skills, or an enthusiast \nexploring Python, this handbook will definitely be helpful. Python\'s simplic

## Embedding and storing in vectorDB


In [10]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingHead:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        
        """ Initialize the embedding head
        
        Args:
            model_name: HuggingFace model name for sentence embeddings """
            
        self.model_name = model_name
        self.model = None
        self._load_model()              

    def _load_model(self):
        
        """Load the SentenceTransformer model"""
        
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        
        """ Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim) """
        
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embeddings=EmbeddingHead()        ##To initialize/start the embedding head
embeddings

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7029.75it/s]


Model loaded successfully. Embedding dimension: 384


## VectorDB Store

In [12]:
class VectorStore:
   
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name : str = "pdf_documents", persist_directory : str = "../data/vector_store"):
        
        """ We will initialize the vector store
        
        Args:
            collection_name :- Name of the ChromaDB collection
            persist_directory :- Directory to persist the Vector store - (Store embeddings on hard disk)"""
            
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)              # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            self.collection = self.client.get_or_create_collection(         # Get or create collection
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"          # Generate unique ID
            ids.append(doc_id)

            metadata = dict(doc.metadata)                       # Prepare metadata
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)             # Document content
            
            embeddings_list.append(embedding.tolist())          # Embedding
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
         

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 515


In [13]:
chunks


[Document(metadata={'producer': '3.0.10 (5.0.18)', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-06-16T02:12:11+05:30', 'author': 'Harry Codes', 'moddate': '2024-06-15T22:44:10+02:00', 'source': '../data/pdf_files/The Ultimate Python Handbook.pdf', 'total_pages': 61, 'page': 1, 'page_label': '2', 'source_file': 'The Ultimate Python Handbook.pdf', 'file_type': 'pdf'}, page_content='1 \n \nPREFACE  \nWelcome to the “Ultimate Python Programming Handbook," your comprehensive guide to \nmastering Python programming. This handbook is designed for beginners and anyone looking to \nstrengthen their foundational knowledge of Python, a versatile and user-friendly programming \nlanguage. \nPURPOSE AND AUDIENCE  \nThis handbook aims to make programming accessible and enjoyable for everyone. Whether \nyou\'re a student new to coding, a professional seeking to enhance your skills, or an enthusiast \nexploring Python, this handbook will definitely be helpful. Python\'s simplic

In [14]:
### Convert text into embeddings 
text = [doc.page_content for doc in chunks]

### Generate embeddings
embeds = embeddings.generate_embeddings(text)       #embeds is declared now and embeddings is the variable defined above after performing embedding step and generate_embeddings is a command (like STL)

### Store in the vectorDB
vectorstore.add_documents(chunks, embeds)

Generating embeddings for 515 texts...


Batches: 100%|██████████| 17/17 [00:06<00:00,  2.68it/s]


Generated embeddings with shape: (515, 384)
Adding 515 documents to vector store...
Successfully added 515 documents to vector store
Total documents in collection: 1030


## RAG Retriever Pipeline 

In [16]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embeddings: EmbeddingHead):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embeddings: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embeddings = embeddings

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embeddings.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore, embeddings)

In [17]:
rag_retriever

In [21]:
rag_retriever.retrieve("Anaconda is a distribution package")

Retrieving documents for query: 'Anaconda is a distribution package'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_ea8f6283_149',
  'content': 'packages you need for scientific computing. With Anaconda you typically get\nthe same features as with MATLAB.\n2.3 Anaconda\nAnaconda is a distribution package, where you get Python compiler, Python\npackages and the Spyder editor, all in one package.\nAnaconda includes Python, the Jupyter Notebook, and other commonly used\npackages for scientific computing and data science.\n21',
  'metadata': {'file_type': 'pdf',
   'page': 23,
   'trapped': '/False',
   'content_length': 375,
   'doc_index': 149,
   'source': '../data/pdf_files/Python Programming.pdf',
   'total_pages': 143,
   'source_file': 'Python Programming.pdf',
   'producer': 'pdfTeX-1.40.28',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1',
   'creationdate': '2026-06-12T10:51:46+00:00',
   'moddate': '2026-06-12T10:51:46+00:00',
   'page_label': '24',
   'creator': 'TeX'},
  'similarity_score': 0.3242809772491455,
  'di